# Context Selection for Multi-Hop QA: Exploration and Baseline Demo
**Course**: AI Tools for Research (Lab 02 / Paper 2)  
**Author**: Pranav Kumar Gupta (MHM2026012@iiita.ac.in), IIIT Allahabad  
**Project**: Fixed-k versus Score-Gap Adaptive-k Proxy Context Selection for Multi-Hop QA  

This notebook demonstrates the end-to-end data loading, retrieval scoring, context selection algorithms (Fixed-k vs. Score-Gap Adaptive-k Proxy), and metric calculations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
from src.data import load_and_validate_dataset
from src.retrieval import BM25Retriever, TFIDFRetriever
from src.selection import select_fixed_k, select_adaptive_k
from src.evaluation import compute_retrieval_metrics

print("Environment and modules successfully imported.")

## 1. Load and Validate Sample Questions
Load verified questions from the HotpotQA distractor validation split (each with exactly 10 candidate paragraphs).

In [ ]:
records, summary = load_and_validate_dataset(n_samples=5)
print(f"Loaded {len(records)} verified records.")
sample = records[0]
print("Sample Question:", sample["question"])
print("Question Type:", sample["type"])
print("Gold Titles:", sample["gold_titles"])

## 2. Compare BM25 vs TF-IDF Scoring on Sample Question

In [ ]:
bm25 = BM25Retriever()
tfidf = TFIDFRetriever()

bm25_ranked, bm25_scores = bm25.rank(sample["question"], sample["candidates"])
tfidf_ranked, tfidf_scores = tfidf.rank(sample["question"], sample["candidates"])

print("BM25 Top-3 Titles:", [sample["candidates"][i]["title"] for i in bm25_ranked[:3]])
print("TF-IDF Top-3 Titles:", [sample["candidates"][i]["title"] for i in tfidf_ranked[:3]])

## 3. Score-Gap Adaptive-k Proxy Selection Cutoff
Evaluate the first discrete drops delta s_i = s_i - s_{i+1} to identify the cutoff k*.

In [ ]:
selected_idx, k_star = select_adaptive_k(bm25_ranked, bm25_scores, max_k=10, fallback_k=3)
gold_idx = [i for i, c in enumerate(sample["candidates"]) if c["is_gold"]]
metrics = compute_retrieval_metrics(selected_idx, gold_idx, sample["candidates"])

print(f"Selected k*: {k_star}")
print("Selected Titles:", [sample["candidates"][i]["title"] for i in selected_idx])
print("Evidence Recall:", metrics["evidence_recall"])
print("Full Coverage:", metrics["full_coverage"])
print("Selected Tokens:", metrics["selected_tokens"])